# 01 · Data pipeline
SEC 10-K filings (EDGAR) → clean text → `RecursiveCharacterTextSplitter(500, 50)` → MiniLM embeddings (d = 384, L2-normalised) → FAISS index.  Rebuild with `python scripts/fetch_financebench_filings.py`, `python scripts/build_financebench_dataset.py` and `python -m src.embeddings`.

In [1]:
import sys, json, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 120)

In [2]:
from src.embeddings import load_chunks, load_queries
chunks = pd.DataFrame(load_chunks()); queries = pd.DataFrame(load_queries())
print(len(chunks), 'chunks from', chunks.doc_name.nunique(), 'filings')
chunks.groupby(['source','company']).agg(filings=('doc_name','nunique'), chunks=('text','size'))

52943 chunks from 50 filings


filings  chunks
source                        company                              
FinanceBench 10-K (EDGAR)     3M                          2    2519
                              AES Corporation             1    2281
                              Activision Blizzard         1    1096
                              Amazon                      2    1182
                              Amcor                       2    1849
                              American Express            1    1516
                              American Water Works        3    3559
                              Best Buy                    3    2368
                              CVS Health                  2    3031
                              Coca-Cola                   3    4330
                              Corning                     3    2568
                              Costco                      1     567
                              JPMorgan                    1    2944
                              Johnson & Johnson           1    1175
                              Kraft Heinz                 1    1282
                              Lockheed Martin             3    3055
                              Microsoft                   2    1585
                              Nike                        4    3194
                              PepsiCo                     2    1777
                              Pfizer                      1    1678
                              Ulta Beauty                 1     695
                              Walmart                     3    2324
Phase 1 big-tech 10-K (EDGAR) Alphabet Inc.               1     920
                              Amazon.com Inc.             1     754
                              Apple Inc.                  1     555
                              Meta Platforms Inc.         1    1305
                              Microsoft Corp.             1     853
                              NVIDIA Corp.                1     916
                              Tesla Inc.                  1    1065

In [3]:
chunks.char_count.describe()

count    52943.000000
mean       363.719925
std        106.301327
min        100.000000
25%        288.000000
50%        389.000000
75%        456.000000
max        499.000000
Name: char_count, dtype: float64

In [4]:
for t in chunks.sample(3, random_state=1).text: print(t, '\n---')

A 10% change in the level of interest rates would not have a material impact on the fair value of our outstanding debt at December 31, 2021. 
---
• These relate to 3M's respirator mask/asbestos, PFAS-related other environmental, and Combat Arms Earplugs matters (as discussed in Note 16). Net costs include the impacts of any changes in accrued liabilities, external legal fees, and insurance recoveries, along with associated tax impacts. 
---
Changes in applicable laws or regulations or evolving interpretations thereof, including increased or additional regulations to limit carbon dioxide and other greenhouse gas emissions as a result of concern over climate change; to discourage the use of plastic materials, including regulations relating to recovery and/or disposal of plastic bottles and other packaging materials due to environmental concerns; 
---


## Evaluation question sets
| Label | Source |
|---|---|
| ID | FinanceBench questions on knowledge-base companies' 10-Ks (evidence verified in our text) |
| Near-OOD | FinanceBench questions on held-out companies (no filing indexed) |
| Near-OOD adv. | counterfactual questions built by entity perturbation (stress test) |
| Far-OOD | Open Trivia Database |

In [5]:
display(queries.groupby(['label','source']).size().rename('questions').to_frame())
queries.groupby('label').head(3)[['query_id','label','query_text']]

,,questions
label,source,
FAR_OOD,Open Trivia Database,100
IN_DISTRIBUTION,FinanceBench,68
NEAR_OOD,FinanceBench,43
NEAR_OOD_ADV,Counterfactual (entity perturbation),100


,query_id,label,query_text
0,ID_0001,IN_DISTRIBUTION,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying o...
1,ID_0002,IN_DISTRIBUTION,Assume that you are a public equities analyst. Answer the following question by primarily using information that is ...
2,ID_0003,IN_DISTRIBUTION,Is 3M a capital-intensive business based on FY2022 data?
7,NEAR_0001,NEAR_OOD,You are an investment banker and your only resource(s) to answer the following question is (are): the statement of f...
8,NEAR_0002,NEAR_OOD,What is Adobe's year-over-year change in unadjusted operating income from FY2015 to FY2016 (in units of percents and...
9,NEAR_0003,NEAR_OOD,What is the FY2017 operating cash flow ratio for Adobe? Operating cash flow ratio is defined as: cash from operation...
111,ADV_0001,NEAR_OOD_ADV,What was Apple's total net revenue from the Apple Car Titan project in fiscal year 2024?
112,ADV_0002,NEAR_OOD_ADV,How much profit did Apple report from its holographic eyewear subscription service?
113,ADV_0003,NEAR_OOD_ADV,What were Apple's capital expenditures for nuclear fusion powered data centers in 2023?
211,FAR_0001,FAR_OOD,The character Wario was first introduced in which game of the Mario video game franchise?


In [6]:
rep = json.load(open('data/financebench/dataset_report.json', encoding='utf-8'))
print('held-out companies:', rep['held_out_companies'])
pd.Series([e['reason'].split(' (')[0] for e in rep['excluded_questions']]).value_counts().rename('excluded FinanceBench questions')

held-out companies: ['AMD', 'Adobe', 'Block', 'Boeing', 'General Mills', 'MGM Resorts', 'Netflix', 'Paypal', 'Verizon']


KB company, but the cited document is not a 10-K    34
evidence not found in extracted filing               5
Name: excluded FinanceBench questions, dtype: int64

## Embedding geometry

In [7]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from src.plotting import SPLIT_COLORS, SPLIT_NAMES
X = np.load('data/corpus_embeddings.npy'); Q = np.load('data/query_embeddings.npy')
pca = PCA(2, random_state=0).fit(X)
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(*pca.transform(X[::5]).T, s=3, c='#c3c2b7', label='corpus chunks (every 5th)')
labels = queries.label.values
for l in SPLIT_COLORS:
    ax.scatter(*pca.transform(Q[labels==l]).T, s=16, c=SPLIT_COLORS[l], edgecolor='white', lw=.4, label=SPLIT_NAMES[l])
ax.legend(fontsize=8); ax.set_title('PCA of MiniLM embeddings'); plt.show()

C:\Users\Sai Krishna\AppData\Local\Temp\ipykernel_39852\2530895775.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(fontsize=8); ax.set_title('PCA of MiniLM embeddings'); plt.show()
